In [1]:
from pathlib import Path
import sys
import zipfile

import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.feature_engineering import prepare_features

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SUBMISSION_DIR = PROJECT_ROOT / "submission"
MODELS_DIR = PROJECT_ROOT / "models"

SUBMISSION_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

In [2]:
train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

X_test = pd.read_parquet(
    PROCESSED_DIR / "X_test_canonical.parquet"
)

TARGET_COLUMN = "Цена"

X_train_features = prepare_features(
    train.drop(columns=[TARGET_COLUMN])
)

X_test_features = prepare_features(X_test)

y = train[TARGET_COLUMN].copy()

print("Train:", X_train_features.shape)
print("Test:", X_test_features.shape)
print("Target:", y.shape)

Train: (8340, 29)
Test: (8341, 29)
Target: (8340,)


In [3]:
RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

ridge_feature_columns = [
    column
    for column in X_train_features.columns
    if column not in EXCLUDED_COLUMNS
]

numeric_columns = [
    "Год выпуска",
    "Оценка эксперта",
    "Количество владельцев",
    "Пробег_число",
    "Расход_л_на_100км",
    "Двигатель_цилиндры",
    "Двигатель_объём_л",
    "Двери_число",
    "Кресла_число",
]

categorical_columns = [
    column
    for column in ridge_feature_columns
    if column not in numeric_columns
]

X_train_ridge = X_train_features[ridge_feature_columns].copy()
X_test_ridge = X_test_features[ridge_feature_columns].copy()

for column in categorical_columns:
    X_train_ridge[column] = (
        X_train_ridge[column]
        .astype("object")
        .where(X_train_ridge[column].notna(), np.nan)
    )

    X_test_ridge[column] = (
        X_test_ridge[column]
        .astype("object")
        .where(X_test_ridge[column].notna(), np.nan)
    )

assert list(X_train_ridge.columns) == list(X_test_ridge.columns)
assert len(X_train_ridge) == 8340
assert len(X_test_ridge) == 8341
assert "Предложение" not in X_train_ridge.columns
assert "car_id" not in X_train_ridge.columns

In [4]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_columns),
        ("cat", categorical_pipeline, categorical_columns),
    ],
    remainder="drop",
)

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=0.1, solver="lsqr")),
    ]
)

final_ridge_model = TransformedTargetRegressor(
    regressor=ridge_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

final_ridge_model.fit(X_train_ridge, y)

test_predictions = final_ridge_model.predict(X_test_ridge)
test_predictions = np.maximum(test_predictions, 1)

print(pd.Series(test_predictions).describe())
print("NaN predictions:", np.isnan(test_predictions).sum())
print("Inf predictions:", np.isinf(test_predictions).sum())
print("Predictions <= 0:", (test_predictions <= 0).sum())

count    8.341000e+03
mean     3.636137e+04
std      3.677116e+04
min      4.707120e+02
25%      1.918621e+04
50%      2.928936e+04
75%      4.374142e+04
max      1.561766e+06
dtype: float64
NaN predictions: 0
Inf predictions: 0
Predictions <= 0: 0


Создаём файл в формате baseline:

In [5]:
submission = pd.DataFrame(
    {
        "Цена": test_predictions,
    }
)

assert len(submission) == len(X_test)
assert submission["Цена"].notna().all()
assert np.isfinite(submission["Цена"]).all()
assert (submission["Цена"] > 0).all()

submission_path = SUBMISSION_DIR / "submission.csv"

submission.to_csv(
    submission_path,
    index=False,
)

display(submission.head())

,Цена
0,31228.639983
1,25666.012562
2,39336.773627
3,52400.246519
4,24291.746525


In [6]:
model_path = MODELS_DIR / "ridge_log_target_alpha_0_1.joblib"

joblib.dump(
    final_ridge_model,
    model_path,
)

zip_path = SUBMISSION_DIR / "submission.zip"

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as zipf:
    zipf.write(
        submission_path,
        arcname="submission.csv",
    )

with zipfile.ZipFile(zip_path, "r") as zipf:
    print("Файлы внутри архива:", zipf.namelist())

print("CSV:", submission_path)
print("ZIP:", zip_path)

Файлы внутри архива: ['submission.csv']
CSV: C:\temp\shift_ml\submission\submission.csv
ZIP: C:\temp\shift_ml\submission\submission.zip


In [7]:
check_submission = pd.read_csv(
    SUBMISSION_DIR / "submission.csv"
)

print(check_submission.shape)
print(check_submission.columns.tolist())
display(check_submission.head())

(8341, 1)
['Цена']


,Цена
0,31228.639983
1,25666.012562
2,39336.773627
3,52400.246519
4,24291.746525
